In [4]:
import cv2
import numpy as np

In [5]:
# Abrir el archivo de video
cap = cv2.VideoCapture("Trafico_3.mp4")

# Variables
contador_carril_1 = 0
contador_carril_2 = 0
contador_carril_3 = 0
contador_carril_4 = 0

# Coordenadas para las regiones de interés (ROI)
roi_carril_1_x, roi_carril_1_y = 420, 80
roi_carril_2_x, roi_carril_2_y = 420, 150
roi_carril_3_x, roi_carril_3_y = 420, 280
roi_carril_4_x, roi_carril_4_y = 420, 340

roi_width, roi_height = 50, 45

umbral_movimiento = 50
cooldown_frames = 50
cooldown_1, cooldown_2, cooldown_3, cooldown_4 = 0, 0, 0, 0

In [6]:
# Crear el sustractor de fondo
fgbg = cv2.createBackgroundSubtractorMOG2(detectShadows=True)

while True:
    retVal, frame = cap.read()
    if not retVal:
        break

    frame_resized = cv2.resize(frame, (640, 480))
    fgmask = fgbg.apply(frame_resized)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)
    fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_CLOSE, kernel)

    contornos, _ = cv2.findContours(fgmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Actualizar cooldowns
    cooldown_1 = max(0, cooldown_1 - 1)
    cooldown_2 = max(0, cooldown_2 - 1)
    cooldown_3 = max(0, cooldown_3 - 1)
    cooldown_4 = max(0, cooldown_4 - 1)

    for cnt in contornos:
        if cv2.contourArea(cnt) < umbral_movimiento:
            continue

        (x, y, w, h) = cv2.boundingRect(cnt)
        centroid = (x + w // 2, y + h // 2)

        if cooldown_1 == 0 and roi_carril_1_x <= centroid[0] <= roi_carril_1_x + roi_width and roi_carril_1_y <= centroid[1] <= roi_carril_1_y + roi_height:
            contador_carril_1 += 1
            cv2.rectangle(frame_resized, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cooldown_1 = cooldown_frames

        elif cooldown_2 == 0 and roi_carril_2_x <= centroid[0] <= roi_carril_2_x + roi_width and roi_carril_2_y <= centroid[1] <= roi_carril_2_y + roi_height:
            contador_carril_2 += 1
            cv2.rectangle(frame_resized, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cooldown_2 = cooldown_frames

        elif cooldown_3 == 0 and roi_carril_3_x <= centroid[0] <= roi_carril_3_x + roi_width and roi_carril_3_y <= centroid[1] <= roi_carril_3_y + roi_height:
            contador_carril_3 += 1
            cv2.rectangle(frame_resized, (x, y), (x + w, y + h), (255, 20, 147), 2)
            cooldown_3 = cooldown_frames

        elif cooldown_4 == 0 and roi_carril_4_x <= centroid[0] <= roi_carril_4_x + roi_width and roi_carril_4_y <= centroid[1] <= roi_carril_4_y + roi_height:
            contador_carril_4 += 1
            cv2.rectangle(frame_resized, (x, y), (x + w, y + h), (255, 255, 0), 2)
            cooldown_4 = cooldown_frames

    # Dibujar los rectángulos de los carriles
    cv2.rectangle(frame_resized, (roi_carril_1_x, roi_carril_1_y), (roi_carril_1_x + roi_width, roi_carril_1_y + roi_height), (255, 0, 0), 2)
    cv2.rectangle(frame_resized, (roi_carril_2_x, roi_carril_2_y), (roi_carril_2_x + roi_width, roi_carril_2_y + roi_height), (255, 0, 0), 2)
    cv2.rectangle(frame_resized, (roi_carril_3_x, roi_carril_3_y), (roi_carril_3_x + roi_width, roi_carril_3_y + roi_height), (255, 0, 0), 2)
    cv2.rectangle(frame_resized, (roi_carril_4_x, roi_carril_4_y), (roi_carril_4_x + roi_width, roi_carril_4_y + roi_height), (255, 0, 0), 2)
    
    # Dibujar los contadores
    def draw_text_with_bg(text, pos, color_bg, color_text):
        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)[0]
        cv2.rectangle(frame_resized,
                      (pos[0] - 5, pos[1] - text_size[1] - 5),
                      (pos[0] + text_size[0] + 5, pos[1] + 5),
                      color_bg, -1)
        cv2.putText(frame_resized, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 1, color_text, 2)

    draw_text_with_bg(f'Carril 1: {contador_carril_1}', (10, 30), (0, 255, 0), (255, 255, 255))
    draw_text_with_bg(f'Carril 2: {contador_carril_2}', (10, 70), (0, 0, 255), (255, 255, 255))
    draw_text_with_bg(f'Carril 3: {contador_carril_3}', (10, 110), (255, 20, 147), (255, 255, 255))
    draw_text_with_bg(f'Carril 4: {contador_carril_4}', (10, 150), (255, 255, 0), (255, 255, 255))

    cv2.imshow('Detección de vehículos', frame_resized)
    # cv2.imshow('Sustracción de fondo', fgmask)

    if cv2.waitKey(15) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()
cap.release()
